In [2]:
import pandas as pd

flowers = pd.read_csv(
    r"C:\Users\leoma\OneDrive\Documents\BasicPyTorchDemo\iris_demo.csv"
)

print(flowers.head())

print(flowers.shape)

print(flowers.dtypes)

   sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm species
0              5.1             3.5              1.4             0.2  setosa
1              4.9             3.0              1.4             0.2  setosa
2              4.7             3.2              1.3             0.2  setosa
3              4.6             3.1              1.5             0.2  setosa
4              5.0             3.6              1.4             0.2  setosa
(150, 5)
sepal_length_cm    float64
sepal_width_cm     float64
petal_length_cm    float64
petal_width_cm     float64
species                str
dtype: object


In [3]:
from sklearn.model_selection import train_test_split

train_all, test = train_test_split(
    flowers,
    test_size=0.20,
    stratify=flowers["species"],
    random_state=42
)

train, val = train_test_split(
    train_all,
    test_size=0.25,
    stratify=train_all["species"],
    random_state=42
)

print(
    len(train),
    len(val),
    len(test)
)

90 30 30


In [4]:
print(train.isna().sum())

print(train.describe())

print(
    train["species"].value_counts()
)

print(
    train.groupby("species")[
        [
            "petal_length_cm",
            "petal_width_cm"
        ]
    ].median()
)

majority = (
    train["species"]
    .value_counts(
        normalize=True
    )
    .max()
)

print(
    "Majority baseline:",
    round(majority, 3)
)

sepal_length_cm    0
sepal_width_cm     0
petal_length_cm    0
petal_width_cm     0
species            0
dtype: int64
       sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm
count        90.000000       90.000000        90.000000       90.000000
mean          5.836667        3.013333         3.751111        1.197778
std           0.866149        0.416297         1.772083        0.761279
min           4.300000        2.000000         1.100000        0.100000
25%           5.100000        2.725000         1.600000        0.300000
50%           5.750000        3.000000         4.250000        1.300000
75%           6.400000        3.200000         5.100000        1.800000
max           7.900000        3.900000         6.900000        2.500000
species
virginica     30
setosa        30
versicolor    30
Name: count, dtype: int64
            petal_length_cm  petal_width_cm
species                                    
setosa                 1.50            0.20
versicolor       

## Training Data Exploration

The training dataset contains 90 flowers.

Each species has 30 training examples.

The majority baseline is approximately 0.333 because the classes are balanced.

Versicolor and virginica appear more similar than setosa based on their petal measurements.

In [1]:
import torch

torch.manual_seed(42)

x = torch.tensor(
    3.0,
    requires_grad=True
)

loss_example = (
    x - 1
) ** 2

loss_example.backward()

print(
    loss_example.item(),
    x.grad.item()
)

4.0 4.0


## Tensor and Gradient

A tensor stores numerical values that can be used in neural network calculations.

The gradient shows the direction and size of the change needed to reduce the loss.

At x = 3, a small decrease in x would reduce the loss value.

In [2]:
from torch import nn

probe = nn.Linear(
    4,
    8
)

print(
    "weight shape:",
    probe.weight.shape
)

print(
    "bias shape:",
    probe.bias.shape
)

print(
    "parameter count:",
    sum(
        p.numel()
        for p in probe.parameters()
    )
)

one_flower = torch.tensor(
    [2.0, 1.0, 0.5, -1.0]
)

print(
    "one flower produces:",
    probe(one_flower).shape
)

weight shape: torch.Size([8, 4])
bias shape: torch.Size([8])
parameter count: 40
one flower produces: torch.Size([8])


## Linear Layer

The layer accepts four input measurements and produces eight output values.

The layer contains 32 weights and eight bias values for a total of 40 parameters.

The outputs are random because the network has not been trained.

In [3]:
z = torch.tensor(
    [-2.0, 0.0, 2.0]
)

for name, activation in [
    ("ReLU", nn.ReLU()),
    (
        "Leaky ReLU",
        nn.LeakyReLU(
            negative_slope=0.01
        )
    ),
    ("sigmoid", nn.Sigmoid()),
    ("tanh", nn.Tanh())
]:

    values = (
        torch.round(
            activation(z) * 100
        ) / 100
    )

    print(
        name,
        values.tolist()
    )

ReLU [0.0, 0.0, 2.0]
Leaky ReLU [-0.019999999552965164, 0.0, 2.0]
sigmoid [0.11999999731779099, 0.5, 0.8799999952316284]
tanh [-0.9599999785423279, 0.0, 0.9599999785423279]


```markdown
## Activation Functions

ReLU turns negative values into zero and leaves positive values unchanged.

Leaky ReLU allows a small negative value instead of zero.

Sigmoid produces values between 0 and 1.

Tanh produces values between -1 and 1.
```

In [4]:
scores = torch.tensor(
    [1.2, -0.4, 2.1]
)

print(
    "predicted class ID:",
    scores.argmax().item()
)

print(
    "illustrative probabilities:",
    torch.softmax(
        scores,
        dim=0
    )
)

predicted class ID: 2
illustrative probabilities: tensor([0.2731, 0.0551, 0.6717])


```markdown
## Output Scores

The output layer produces one score for each class.

The highest score determines the predicted class.

Softmax can be used to display probabilities, but CrossEntropyLoss expects raw logits.
```

In [6]:
import pandas as pd

flowers = pd.read_csv(
    r"C:\Users\leoma\OneDrive\Documents\BasicPyTorchDemo\iris_demo.csv"
)

print(flowers.shape)

(150, 5)


In [7]:
from sklearn.model_selection import train_test_split

train_all, test = train_test_split(
    flowers,
    test_size=0.20,
    stratify=flowers["species"],
    random_state=42
)

train, val = train_test_split(
    train_all,
    test_size=0.25,
    stratify=train_all["species"],
    random_state=42
)

print(
    len(train),
    len(val),
    len(test)
)

90 30 30


In [8]:
import torch
from sklearn.preprocessing import StandardScaler
import numpy as np

features = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
]

classes = sorted(
    train["species"].unique()
)

label_to_id = {
    name: i
    for i, name in enumerate(classes)
}

scaler = StandardScaler().fit(
    train[features]
)

def as_tensors(frame):

    X = torch.tensor(
        scaler.transform(
            frame[features]
        ),
        dtype=torch.float32
    )

    y = torch.tensor(
        frame["species"]
        .map(label_to_id)
        .to_numpy(),
        dtype=torch.long
    )

    return X, y

X_train, y_train = as_tensors(train)

X_val, y_val = as_tensors(val)

print(
    X_train.shape,
    y_train.shape,
    classes
)

torch.Size([90, 4]) torch.Size([90]) ['setosa', 'versicolor', 'virginica']


In [9]:
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

loader = DataLoader(
    TensorDataset(
        X_train,
        y_train
    ),
    batch_size=16,
    shuffle=True
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

for epoch in range(100):

    model.train()

    for xb, yb in loader:

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        optimizer.step()

    if epoch in (0, 24, 49, 99):

        print(
            epoch + 1,
            round(
                loss.item(),
                3
            )
        )

1 0.952
25 0.125
50 0.03
100 0.012


## Neural Network Training

The network accepts four flower measurements as input.

The hidden layer contains eight neurons and uses the ReLU activation function.

CrossEntropyLoss measures prediction error and Adam updates the weights.

The training loss generally decreases as the network learns.

In [10]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix
)

def evaluate(X, y):

    model.eval()

    with torch.inference_mode():

        pred = model(X).argmax(dim=1)

    return (
        accuracy_score(
            y.numpy(),
            pred.numpy()
        ),
        confusion_matrix(
            y.numpy(),
            pred.numpy(),
            labels=range(3)
        )
    )

val_acc, val_cm = evaluate(
    X_val,
    y_val
)

print(
    "Validation accuracy:",
    round(
        val_acc,
        3
    )
)

print(val_cm)

Validation accuracy: 0.933
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


In [11]:
smaller_model = nn.Sequential(
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 3)
)

small_optimizer = torch.optim.Adam(
    smaller_model.parameters(),
    lr=0.01
)

for epoch in range(100):

    smaller_model.train()

    for xb, yb in loader:

        small_optimizer.zero_grad()

        logits = smaller_model(xb)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        small_optimizer.step()

smaller_model.eval()

with torch.inference_mode():

    small_pred = (
        smaller_model(X_val)
        .argmax(dim=1)
    )

small_acc = accuracy_score(
    y_val.numpy(),
    small_pred.numpy()
)

print(
    "Four-neuron validation accuracy:",
    round(small_acc, 3)
)

print(
    "Eight-neuron validation accuracy:",
    round(val_acc, 3)
)

Four-neuron validation accuracy: 0.933
Eight-neuron validation accuracy: 0.933


## Hidden Layer Comparison

The four-neuron and eight-neuron models were compared using the same training and validation data.

Only the hidden layer size changed, making the comparison fair.

If both models perform similarly, the smaller model may be preferred because it is simpler.

In [12]:
chosen_size = (
    4
    if small_acc >= val_acc
    else 8
)

final_model = nn.Sequential(
    nn.Linear(4, chosen_size),
    nn.ReLU(),
    nn.Linear(chosen_size, 3)
)

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=0.01
)

X_develop, y_develop = as_tensors(
    train_all
)

loader_final = DataLoader(
    TensorDataset(
        X_develop,
        y_develop
    ),
    batch_size=16,
    shuffle=True
)

for epoch in range(100):

    final_model.train()

    for xb, yb in loader_final:

        final_optimizer.zero_grad()

        logits = final_model(xb)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        final_optimizer.step()

X_test, y_test = as_tensors(test)

final_model.eval()

with torch.inference_mode():

    test_pred = (
        final_model(X_test)
        .argmax(dim=1)
    )

test_acc = accuracy_score(
    y_test.numpy(),
    test_pred.numpy()
)

print(
    "Final test accuracy:",
    round(test_acc, 3)
)

new_flower = torch.tensor(
    scaler.transform(
        [[5.1, 3.5, 1.4, 0.2]]
    ),
    dtype=torch.float32
)

with torch.inference_mode():

    prediction = (
        final_model(new_flower)
        .argmax(dim=1)
        .item()
    )

print(
    "New flower prediction:",
    classes[prediction]
)

Final test accuracy: 1.0
New flower prediction: setosa


C:\Users\leoma\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [13]:
torch.save(
    final_model.state_dict(),
    "iris_pytorch_weights.pt"
)

reloaded = nn.Sequential(
    nn.Linear(4, chosen_size),
    nn.ReLU(),
    nn.Linear(chosen_size, 3)
)

reloaded.load_state_dict(
    torch.load(
        "iris_pytorch_weights.pt",
        map_location="cpu"
    )
)

reloaded.eval()

with torch.inference_mode():

    sample_prediction = (
        reloaded(X_test[:1])
        .argmax(dim=1)
        .item()
    )

print(
    "Reloaded prediction:",
    classes[sample_prediction]
)

Reloaded prediction: setosa
